# How the agent talks to us — and how we talk back

**Focus:** voice in, voice out, and the **live transcript** Chatbot.

**Open beside this:**
- `app.py` → `iv_audio`, `iv_answer_audio`, `_ask_question`, `interview_submit`, `_turns_to_messages`
- `careercrew/audio.py` → `speak`, `transcribe`


## Say this (20 seconds)

> When the interviewer asks a question, we take the **same text string**, synthesize speech with TTS for the ear, and append that string as an interviewer turn for the eye. When I answer, Gradio saves my mic to a temp file; Whisper turns it into text (or I type). Only that text goes to the evaluator. Then we append my answer to the transcript and speak the next question the same way. **Voice and transcript are always the same underlying text — audio is just I/O around it.**


## Board diagram

```
AGENT SPEAKS                          YOU ANSWER
───────────                           ──────────
question.text                         mic → Gradio temp filepath
     │                                      │
     ├─► audio.speak() → .mp3               ├─► audio.transcribe() → TEXT
     │   → gr.Audio(autoplay)               │         OR typed Textbox
     │                                      │
     └─► append_turn(INTERVIEWER)           └─► append_turn(CANDIDATE)
              │                                      │
              └────────── Chatbot ←──────────────────┘
                   (_turns_to_messages)

Chat LLM / evaluate_answer sees TEXT ONLY — never the audio bytes.
```


## 1. UI widgets (Gradio) — where the mic and speaker live

File: `app.py` (Mock interview tab, ~1363–1378)

| Widget | Role |
|--------|------|
| `iv_audio` | Interviewer player (`autoplay=True`, not interactive) |
| `iv_answer_audio` | Your mic (`sources=["microphone"]`, `type="filepath"`) |
| `iv_answer_text` | Optional typed fallback |
| `iv_chatbot` | Live transcript |


In [ ]:
# Exact shapes from app.py (illustrative — not importing Gradio here)

ui = {
    "iv_audio": {
        "label": "Interviewer (listen)",
        "autoplay": True,
        "interactive": False,  # plays TTS mp3 from _ask_question
    },
    "iv_answer_audio": {
        "sources": ["microphone"],
        "type": "filepath",  # Gradio passes a temp path string into interview_submit
        "label": "Record your answer",
    },
    "iv_answer_text": {
        "label": "Or type your answer",
        "optional": True,
    },
    "iv_chatbot": {
        "label": "Interview transcript",
        "fed_by": "_turns_to_messages(session)",
    },
}

for name, cfg in ui.items():
    print(f"{name:16} → {cfg}")


## 2. Agent talks = TTS + transcript turn

`_ask_question` in `app.py` (~1043–1060):

1. `audio.speak(question.text)` → temp `.mp3` (or `None` if no key)
2. `interview.append_turn(..., INTERVIEWER, question.text)`
3. Return path to `iv_audio` **and** `_turns_to_messages(session)` to `iv_chatbot`

`audio.speak` (`careercrew/audio.py`): OpenAI **`tts-1`** → write temp mp3 → return path. On failure → `None` (text-only mode).


In [ ]:
# Simulate what _ask_question does with the transcript (no API calls)

def speak_stub(text: str):
    """Stand-in for audio.speak — real code writes a temp .mp3."""
    print(f"[TTS] would synthesize: {text!r}")
    return "/tmp/careercrew_q_demo.mp3"  # fake path for the demo

turns = []  # like session.turns

def append_turn(role: str, text: str):
    turns.append({"role": role, "text": text})

question_text = "Tell me about a time you owned an API end-to-end."
audio_path = speak_stub(question_text)
append_turn("interviewer", question_text)

print("audio_path →", audio_path)
print("turns      →", turns)


## 3. You talk = mic filepath → Whisper → text

`interview_submit` in `app.py` (~1092–1094):

```python
transcript = audio.transcribe(answer_audio) if answer_audio else ""
answer = transcript.strip() or (answer_text or "").strip()
```

- Gradio recorded the mic and passed **`answer_audio`** as a filepath.
- `audio.transcribe` opens the file → OpenAI **`whisper-1`** → string.
- Prefer voice transcript; if empty, use typed text.
- Then `validate_interview_answer` + `evaluate_answer` (chat LLM, **text only**).


In [ ]:
# Simulate STT preference: voice wins over typed text when both exist

def transcribe_stub(path):
    """Stand-in for audio.transcribe — real code calls whisper-1."""
    if not path:
        return ""
    return "I built a FastAPI service, owned deploy, and cut latency 40%."

answer_audio = "/tmp/gradio_mic_xyz.wav"   # Gradio temp filepath
answer_text = "short typed draft"         # optional

transcript = transcribe_stub(answer_audio) if answer_audio else ""
answer = transcript.strip() or (answer_text or "").strip()

print("final answer to LLM →", answer)
print("note: chat model never sees", answer_audio)


## 4. Live transcript sync

`_turns_to_messages` (`app.py` ~926–932):

| Session role | Chatbot role |
|--------------|--------------|
| `INTERVIEWER` | `"assistant"` |
| `CANDIDATE` | `"user"` |

Updated after every ask and every submit so the Chatbot stays synchronized with voice.


In [ ]:
# Same mapping as app._turns_to_messages

def turns_to_messages(turns):
    messages = []
    for turn in turns:
        role = "assistant" if turn["role"] == "interviewer" else "user"
        messages.append({"role": role, "content": turn["text"]})
    return messages

# Continue the mock session from earlier cells
append_turn("candidate", answer)
append_turn("interviewer", "How did you measure that latency improvement?")

chatbot_messages = turns_to_messages(turns)
print("What the Gradio Chatbot receives:\n")
for m in chatbot_messages:
    who = "Interviewer" if m["role"] == "assistant" else "You"
    print(f"  {who}: {m['content']}")


## 5. Failure-tolerant design (say this if asked)

| Failure | Behavior |
|---------|----------|
| No OpenAI key | `audio_available()` false → text-only interview |
| TTS fails | Question still shown as HTML text |
| STT fails / no mic | Typed `iv_answer_text` still works |
| Empty answer | Guardrail rejects; ask again |

Never crash the UI for audio problems (`audio.py` catches exceptions → `""` / `None`).


## Code checklist for the professor

| Step | File | Symbol |
|------|------|--------|
| Mic + speaker UI | `app.py` | `iv_answer_audio`, `iv_audio`, `iv_chatbot` |
| Ask (TTS + turn) | `app.py` | `_ask_question` |
| Submit (STT + eval) | `app.py` | `interview_submit` |
| Transcript map | `app.py` | `_turns_to_messages` |
| Whisper / TTS | `careercrew/audio.py` | `transcribe`, `speak` |
| Persist text | `session_store.py` | `save_interview` |

**Next notebook:** `02_mock_interview_flow.ipynb` (full start → score loop).
